In [1]:
!git clone https://github.com/vinhVVN/Realized-Volatility-Prediction.git

Cloning into 'Realized-Volatility-Prediction'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 88 (delta 31), reused 79 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 50.14 KiB | 5.57 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [2]:
import os
import sys

# 2. Chuyển "tâm vũ trụ" (working directory) vào bên trong thư mục vừa clone
PROJECT_ROOT = '/kaggle/working/Realized-Volatility-Prediction'
os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print(f"Đã chuyển thư mục làm việc đến: {os.getcwd()}")

Đã chuyển thư mục làm việc đến: /kaggle/working/Realized-Volatility-Prediction


In [3]:
!pip install fastparquet pytorch-tabnet -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 44.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.9 MB/s eta 0:00:00


In [4]:
import pandas as pd
import gc
from src.features.feature_pipeline import build_features
from src.data.data_loader import DataBlock

# Đường dẫn tới thư mục dữ liệu TRÊN KAGGLE (Cực kỳ quan trọng)
DATA_DIR = '/kaggle/input/competitions'
RAW_DIR = os.path.join(DATA_DIR, 'optiver-realized-volatility-prediction')

# Tạo thư mục chứa file đầu ra
PROCESSED_DIR = './data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# 1. Đọc file train.csv nền tảng
df_train_full = pd.read_csv(f'{RAW_DIR}/train.csv')
all_stocks = df_train_full['stock_id'].unique()

print(f"Tổng số cổ phiếu cần xử lý: {len(all_stocks)}")

# 2. KỸ THUẬT CHUNKING: Chia nhỏ để trị (Ví dụ: mỗi lần chạy 10 stocks)
chunk_size = 10
total_chunks = (len(all_stocks) // chunk_size) + 1

for i in range(total_chunks):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(all_stocks))
    
    current_stocks = all_stocks[start_idx:end_idx]
    if len(current_stocks) == 0: break
        
    print(f"\n--- Đang xử lý Chunk {i+1}/{total_chunks} (Stocks: {current_stocks.tolist()}) ---")
    
    # Lọc df_train chỉ lấy các mã cổ phiếu trong chunk này
    df_chunk = df_train_full[df_train_full['stock_id'].isin(current_stocks)].reset_index(drop=True)
    
    # Chạy Pipeline
    df_features_chunk = build_features(
        df_train=df_chunk, 
        data_dir=RAW_DIR, 
        block=DataBlock.TRAIN, 
        config_path='./configs/feature_config.yaml'
    )
    
    # Lưu ngay xuống ổ cứng thành file feather riêng lẻ
    save_path = f'{PROCESSED_DIR}/features_chunk_{i+1}.feather'
    df_features_chunk.reset_index(drop=True).to_feather(save_path)
    print(f"Đã lưu: {save_path}")
    
    # Dọn dẹp RAM triệt để trước khi qua vòng lặp mới
    del df_chunk, df_features_chunk
    gc.collect()

print("\n✅ HOÀN TẤT TOÀN BỘ CHUNKS!")

Tổng số cổ phiếu cần xử lý: 112

--- Đang xử lý Chunk 1/12 (Stocks: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]) ---
2026-06-08 01:29:08 | INFO     | feature_pipeline | Đã phát hiện 10 unique stock_id trong df_train. Tiến hành filter dữ liệu đọc vào RAM.
2026-06-08 01:29:08 | INFO     | feature_pipeline | [a) Đọc dữ liệu Book và Trade song song] Bắt đầu...
2026-06-08 01:29:08 | INFO     | data_loader | Đang đọc song song 10 file book train (n_jobs=4)...
2026-06-08 01:29:13 | INFO     | data_loader | Hoàn tất đọc dữ liệu. Kích thước tổng: (12899265, 11)
2026-06-08 01:29:13 | INFO     | data_loader | Đang đọc song song 10 file trade train (n_jobs=4)...
2026-06-08 01:29:13 | INFO     | data_loader | Hoàn tất đọc dữ liệu. Kích thước tổng: (2546496, 6)
2026-06-08 01:29:13 | INFO     | feature_pipeline | [a) Đọc dữ liệu Book và Trade song song] Hoàn thành trong 5.4278 giây.
2026-06-08 01:29:13 | INFO     | feature_pipeline | [b) Tính toán Base Features và Tick Size] Bắt đầu...
2026-06-08 01:30:31 | INFO 

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


2026-06-08 01:53:13 | INFO     | data_loader | Hoàn tất đọc dữ liệu. Kích thước tổng: (13568013, 11)
2026-06-08 01:53:13 | INFO     | data_loader | Đang đọc song song 10 file trade train (n_jobs=4)...
2026-06-08 01:53:13 | INFO     | data_loader | Hoàn tất đọc dữ liệu. Kích thước tổng: (3209976, 6)
2026-06-08 01:53:13 | INFO     | feature_pipeline | [a) Đọc dữ liệu Book và Trade song song] Hoàn thành trong 3.5130 giây.
2026-06-08 01:53:13 | INFO     | feature_pipeline | [b) Tính toán Base Features và Tick Size] Bắt đầu...
2026-06-08 01:54:31 | INFO     | feature_pipeline | [b) Tính toán Base Features và Tick Size] Hoàn thành trong 77.9608 giây.
2026-06-08 01:54:31 | INFO     | feature_pipeline | [c) Merge dữ liệu, tính Tau, Skew Correction và Rank Normalization] Bắt đầu...
2026-06-08 01:54:31 | INFO     | feature_pipeline | [c) Merge dữ liệu, tính Tau, Skew Correction và Rank Normalization] Hoàn thành trong 0.2260 giây.
2026-06-08 01:54:31 | INFO     | feature_pipeline | [d) Xây dựng K

In [5]:
import glob

# Đọc tất cả các file chunk và nối lại thành 1 file duy nhất
all_files = glob.glob(f'{PROCESSED_DIR}/features_chunk_*.feather')
df_final = pd.concat([pd.read_feather(f) for f in all_files], ignore_index=True)

print(f"Kích thước DataFrame cuối cùng: {df_final.shape}")

# Lưu file tổng thể
df_final.to_feather(f'{PROCESSED_DIR}/features_FINAL.feather')
print("Đã gộp xong!")

Kích thước DataFrame cuối cùng: (428932, 574)
Đã gộp xong!
